In [87]:
# Main imports
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import pandas as pd
import nltk

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

In [88]:
# Only run once
import nltk

nltk.download("all")

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to
[nltk_data]    |     C:\Users\SEBASTIAN\AppData\Roaming\nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to
[nltk_data]    |     C:\Users\SEBASTIAN\AppData\Roaming\nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     C:\Users\SEBASTIAN\AppData\Roaming\nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     C:\Users\SEBASTIAN\AppData\Roaming\nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     C:\Users\SEBASTIAN\AppData\Ro

True

In [89]:
import pandas as pd
import emoji

# Load cached emoji translations
# CSV columns: emoji, description_en, description_es, description_pt
emoji_df = pd.read_csv("emoji_translations.csv")
emoji_map = {
    "en": dict(zip(emoji_df["emoji"], emoji_df["description_en"])),
    "es": dict(zip(emoji_df["emoji"], emoji_df["description_es"])),
    "pt": dict(zip(emoji_df["emoji"], emoji_df["description_pt"]))
}

# Helper function for preprocessing using cached translations
def translate_emojis_in_text(text: str, target_lang="es"):
    if target_lang not in emoji_map:
        raise ValueError(f"Unsupported language: {target_lang}")

    emojis_found = emoji.distinct_emoji_list(text)
    
    for em in emojis_found:
        if em in emoji_map[target_lang] and pd.notna(emoji_map[target_lang][em]):
            translated_meaning = emoji_map[target_lang][em]
            text = text.replace(em, translated_meaning)
        else:
            # Fallback: Remove the emoji if no translation is found
            text = text.replace(em, "")
            pass

    return text

In [90]:
import re

def clean_text(text):
    # Remove URLs (http, https, www)
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove special characters (keep letters, numbers, spaces, basic punctuation)
    text = re.sub(r'[^A-Za-z0-9ÁÉÍÓÚáéíóúÑñÜü.,!?;:()\[\]\'" ]+', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [91]:
# create preprocessor class
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

LANGUAGE_MAP = {
    "en": "english",
    "es": "spanish",
    "pt": "portuguese"
}

class Preprocessor:
    def __init__(self, language="en"):
        if language not in LANGUAGE_MAP:
            raise ValueError(f"Unsupported language: {language}")
        
        self.language = language

    def preprocess(self, text: str) -> str:
        # 1. Lowercase first
        lowered = text.lower()
        
        # 2. Translate emojis
        demojized = translate_emojis_in_text(lowered, self.language)

        # 3. Remove URLs & special chars
        cleaned = clean_text(demojized)

        # 4. Tokenize (word level)
        tokens = word_tokenize(cleaned.casefold())

        # 5. Remove stop words
        filtered_tokens = [t for t in tokens if t not in stopwords.words(LANGUAGE_MAP[self.language])]

        # 6. Lemmatize
        lemmatizer = WordNetLemmatizer()
        lemmatized_tokens = [lemmatizer.lemmatize(t) for t in filtered_tokens]

        # 7. Join tokens
        processed = ' '.join(lemmatized_tokens)
        return processed
    def preprocess_dataset(self,df):
        df["text"] = df["text"].apply(self.preprocess)
        return df

In [92]:
# create sentiment analyzer class
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# Model mapping for each language
MODEL_MAP = {
    "en": "finiteautomata/bertweet-base-sentiment-analysis",  # English (social media focused)
    "es": "pysentimiento/robertuito-sentiment-analysis",    # Spanish (tweets)
    "pt": "pysentimiento/bertweet-pt-sentiment"             # Portuguese (tweets)
}

class SentimentAnalyzer:
    def __init__(self, language="en"):
        if language not in MODEL_MAP:
            raise ValueError(f"Unsupported language: {language}")
        
        self.language = language

        # Load model and tokenizer for given language
        model_name = MODEL_MAP[self.language]
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(model_name)

        # Create pipeline
        self.nlp = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

    def predict(self, text: str):
        # Get prediction
        result = self.nlp(text)[0]  # {'label': 'POSITIVE', 'score': 0.99}
        sentiment_score = self.__map_sentiment(result['label'])
        return sentiment_score
    
    # Map model outputs to -1, 0, 1
    def __map_sentiment(self, label: str):
        # Lowercase to be safe
        label = label.casefold()
        if "pos" in label:
            return 1
        elif "neg" in label:
            return -1
        else:
            return 0



In [ ]:
preprocessor = Preprocessor(language="en")
df = pd.read_csv('../data/cyberbullying_dataset_raw.csv')
df_processed = preprocessor.preprocess_dataset(df)
df_processed.to_csv('../data/cyberbullying_dataset.csv', index=False)

In [6]:
analyzer = SentimentAnalyzer(language="en")
df_processed["sentiment"] = df_processed["text"].apply(analyzer.predict)
df_processed = df_processed[df_processed["sentiment"] != 1]
df_processed = df_processed.drop(columns=["sentiment"])
df_processed.to_csv('../data/cyberbullying_dataset.csv', index=False)

In [8]:
df_processed.info()
#label distribution df
df_processed["label"].value_counts().plot(kind='bar')

Text 1: I love this movie!
Prediction: not_cyberbullying

Text 2: Odio esta pelicula!
Prediction: other_cyberbullying

